### Análisis y Procesamiento de Señales
# Trabajo Práctico N° 2 - Simulación de ADC: Muestreo y Cuantización
### Pedro Joaquín Cannavo

## 1. Introducción

En este trabajo práctico se continuó con la simulación de un conversor analógico-digital (**ADC**) uniforme de $B$ bits con un rango de entrada de $\pm V_F = \pm 2\text{ V}$, agregando ahora la etapa de muestreo a una frecuencia $f_s = 1000\text{ Hz}$.

El objetivo principal es analizar cómo se comporta el conversor cuando digitalizamos una señal senoidal a la que se le suma ruido analógico gaussiano, y ver cómo interactúan entre sí las dos fuentes de ruido presentes:
1. El **ruido analógico** que viene junto con la señal a la entrada.
2. El **ruido de cuantización** que genera el propio ADC al redondear los valores continuos a niveles discretos.

Para estudiar esto, se analizaron las señales en el tiempo, se construyó el histograma del error para verificar su distribución uniforme y se representaron los espectros en decibeles tomando como referencia de $0\text{ dB}$ la potencia de la senoidal. Por último, se estudió qué sucede con la relación señal a ruido (SNR) y los pisos de ruido al cambiar la resolución ($B = 4, 8, 16$ bits) y la cantidad de ruido analógico ($k_n = 0.1, 1, 10$).

## 2. Marco Teórico y Parámetros de Diseño

### 2.1 Señal Senoidal y Resolución Frecuencial
Se trabajó con un total de $N = 1000$ muestras y una frecuencia de muestreo de $f_s = 1000\text{ Hz}$, lo que da una duración total de registro de $1$ segundo.

La resolución frecuencial de la FFT viene dada por:

$$\Delta f = \frac{f_s}{N} = \frac{1000\text{ Hz}}{1000} = 1\text{ Hz}$$

Para evitar el desparramo espectral (*spectral leakage*), se eligió una frecuencia para la senoidal que coincida exactamente con el primer bin de la FFT ($f_0 = \Delta f = 1\text{ Hz}$). De esta forma, dentro de la ventana de 1000 muestras entra exactamente un ciclo completo de la señal y toda su energía queda concentrada en ese bin sin ensuciar las frecuencias vecinas.

Para que la señal tenga potencia unitaria ($P_s = 1\text{ W}$) sobre una resistencia de referencia de $R = 1\,\Omega$, la tensión eficaz necesaria es:

$$V_{rms} = \sqrt{P_s \cdot R} = 1\text{ V}$$

y por lo tanto, la amplitud máxima de la senoidal es:

$$A = V_{rms} \sqrt{2} = \sqrt{2}\text{ V} \approx 1.414\text{ V}$$

### 2.2 Cuantización y Ruido de Cuantización
El conversor tiene un rango de entrada de $\pm V_F = \pm 2\text{ V}$. El paso de cuantización $q$ para $B$ bits se calcula como:

$$q = \frac{V_F}{2^B}$$

Para $B = 4$ bits y $V_F = 2\text{ V}$, el paso resultante es $q = 0.125\text{ V}$.

El error de cuantización se define como la diferencia entre la señal cuantizada que sale del ADC y la señal que entra al conversor:

$$e[n] = s_Q[n] - s_R[n]$$

Siempre que la señal no sature y varíe a lo largo de varios niveles, este error se distribuye de manera uniforme en el intervalo $\left[-\frac{q}{2}, +\frac{q}{2}\right]$. Su valor medio es nulo y su potencia teórica viene dada por la varianza de dicha distribución:

$$P_q = \frac{q^2}{12}$$

### 2.3 Ruido Analógico Aditivo
La señal que ingresa al conversor es $s_R[n] = s[n] + n[n]$, donde $n[n]$ es una secuencia de ruido gaussiano incorrelado con media cero y una potencia proporcional al ruido de cuantización:

$$P_n = k_n \cdot P_q$$

El factor $k_n$ permite regular qué tan grande es el ruido analógico respecto al de cuantización. La desviación estándar utilizada para generar el ruido en la simulación es $\sigma_n = \sqrt{P_n}$.

### 2.4 Relación Señal a Ruido (SNR) y Piso de Ruido
A la entrada del conversor, la relación señal a ruido depende únicamente del ruido analógico:

$$\text{SNR}_{\text{antes}} = 10 \log_{10}\left(\frac{P_s}{P_n}\right)$$

A la salida del conversor se suma el ruido introducido por la cuantización, por lo que la potencia total de ruido es la suma de ambas fuentes ($P_n + P_q$):

$$\text{SNR}_{\text{después}} = 10 \log_{10}\left(\frac{P_s}{P_n + P_q}\right)$$

En el espectro en decibeles, como la energía de un ruido blanco se reparte entre los $N/2 = 500$ bines de frecuencias positivas, el nivel medio del piso de ruido queda ubicado en:

$$\text{Piso}_{\text{dB}} = 10 \log_{10}\left(\frac{P_{\text{ruido}}}{N/2}\right)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Parametros generales
N = 1000                  # Cantidad de muestras
fs = 1000.0               # Frecuencia de muestreo [Hz]
VF = 2.0                  # Rango analogico (+/- VF) [V]
R = 1.0                   # Resistencia de referencia [ohm]

# 2. Eje temporal y senal senoidal de 1 W
f0 = fs / N               # f0 = 1 Hz
tt = np.arange(N) / fs
s = np.sqrt(2) * np.sin(2 * np.pi * f0 * tt)
Ps = np.mean(s**2) / R

# 3. Eje de frecuencias (0 a fs/2 = 500 Hz)
ff = np.fft.rfftfreq(N, 1/fs)

# Espectro de la senal pura (pico en 0 dB)
S_orig_db = 10 * np.log10(2 * np.abs(np.fft.rfft(s))**2 / (N**2) + 1e-15)

### 2.5 Función de Simulación del ADC
Definimos una función para simular el proceso completo de digitalización y calcular las potencias, la SNR y los espectros.

In [ ]:
def procesar_adc(B, kn, VF=2.0, N=1000, fs=1000.0):
    """
    Simula el proceso de conversion A/D para B bits y relacion kn.
    """
    # Paso de cuantizacion
    q = VF / (2**B)
    Pq_teorica = (q**2) / 12.0
    Pn_teorica = kn * Pq_teorica
    
    # Generacion de ruido analogico gaussiano
    sigma_n = np.sqrt(Pn_teorica)
    ruido_analogico = np.random.normal(0, sigma_n, N)
    sR = s + ruido_analogico
    
    # Cuantizacion uniforme con saturacion
    sR_clip = np.clip(sR, -VF, VF)
    s_cuantizada = np.round(sR_clip / q) * q
    ruido_cuantizacion = s_cuantizada - sR_clip
    
    # Medicion de potencias muestrales
    Pn_medida = np.mean(ruido_analogico**2)
    Pq_medida = np.mean(ruido_cuantizacion**2)
    
    # Calculo de SNR
    SNR_antes = 10 * np.log10(Ps / Pn_medida)
    SNR_despues = 10 * np.log10(Ps / (Pn_medida + Pq_medida))
    delta_SNR = SNR_antes - SNR_despues
    
    # Espectros en dB
    S_in_db  = 10 * np.log10(2 * np.abs(np.fft.rfft(sR))**2 / (N**2) + 1e-15)
    S_out_db = 10 * np.log10(2 * np.abs(np.fft.rfft(s_cuantizada))**2 / (N**2) + 1e-15)
    
    # Pisos teoricos de ruido
    piso_analog = 10 * np.log10(Pn_teorica / (N / 2))
    piso_digital = 10 * np.log10(Pq_teorica / (N / 2))
    
    return {
        'B': B, 'kn': kn, 'q': q,
        'sR': sR, 's_cuantizada': s_cuantizada, 'ruido_cuantizacion': ruido_cuantizacion,
        'Pq': Pq_teorica, 'Pn': Pn_teorica,
        'Pq_medida': Pq_medida, 'Pn_medida': Pn_medida,
        'SNR_antes': SNR_antes, 'SNR_despues': SNR_despues, 'delta_SNR': delta_SNR,
        'S_in_db': S_in_db, 'S_out_db': S_out_db,
        'piso_analog': piso_analog, 'piso_digital': piso_digital
    }

def graficar_pisos(ax, piso_analog, piso_digital):
    """
    Grafica los pisos teoricos de ruido en dB.
    Si coinciden (kn = 1), alterna guiones rojo/cyan para que se vean ambos.
    """
    lbl_a = f'Piso analog. = {piso_analog:.1f} dB'
    lbl_d = f'Piso digital = {piso_digital:.1f} dB'
    if np.isclose(piso_analog, piso_digital, atol=0.2):
        ax.axhline(piso_analog, color='red', linestyle=(0, (4, 4)), lw=1.6, label=lbl_a)
        ax.axhline(piso_digital, color='cyan', linestyle=(4, (4, 4)), lw=1.6, label=lbl_d)
    else:
        ax.axhline(piso_analog, color='red', linestyle='--', lw=1.3, label=lbl_a)
        ax.axhline(piso_digital, color='cyan', linestyle='--', lw=1.3, label=lbl_d)

## 3. Desarrollo y Experimentación

### 3.1 Inciso a: Experimentación con $B = 4$ bits y $k_n = 1.0$

Para este primer caso se fijó una resolución de 4 bits y una relación $k_n = 1$, lo que significa que la potencia del ruido analógico es igual a la potencia del ruido de cuantización ($P_n = P_q$).

In [ ]:
res_a = procesar_adc(B=4, kn=1.0)

print("="*65)
print(">>> RESULTADOS DEL INCISO A (B = 4 bits, kn = 1.0) <<<")
print(f"Paso de cuantizacion q:             {res_a['q']:.4f} V")
print(f"Potencia teorica Pq:                {res_a['Pq']:.6f} W")
print(f"Potencia teorica Pn:                {res_a['Pn']:.6f} W")
print(f"Piso digital teorico:               {res_a['piso_digital']:.2f} dB")
print(f"Piso analogico teorico:             {res_a['piso_analog']:.2f} dB")
print("-----------------------------------------------------------------")
print(f"SNR antes del ADC:                  {res_a['SNR_antes']:.2f} dB")
print(f"SNR despues del ADC:                {res_a['SNR_despues']:.2f} dB")
print(f"Caida de SNR:                       {res_a['delta_SNR']:.2f} dB (Teorico: ~3.01 dB)")
print("="*65)

In [ ]:
# GRAFICO 1: Dominio del Tiempo
plt.figure(1, figsize=(10, 4.5))
plt.plot(tt, res_a['s_cuantizada'], label=r'$s_Q = Q_{B, V_R}(s_R)$ (ADC out)', color='C0', lw=1.2)
plt.plot(tt, res_a['sR'], label=r'$s_R = s + n$ (ADC in)', color='green', linestyle=':', alpha=0.7)
plt.plot(tt, s, label=r'$s$ (analog)', color='orange', linestyle=':', alpha=0.7)
plt.title(f'Señal muestreada por un ADC de 4 bits - $\pm V_R = {VF:.1f}$ V - q = {res_a["q"]:.3f} V')
plt.xlabel('tiempo [segundos]')
plt.ylabel('Amplitud [V]')
plt.grid(True)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Análisis de la Figura 1 (Tiempo):**
En este gráfico se comparan las tres señales: la senoidal continua original $s(t)$, la señal con ruido analógico $s_R(t)$ y la señal cuantizada $s_Q[n]$ a la salida del ADC. Como el conversor tiene solo 4 bits, el paso de cuantización es relativamente grande ($q = 0.125\text{ V}$), por lo que se pueden ver a simple vista los escalones generados por el redondeo en cada nivel.

In [ ]:
# GRAFICO 2: Densidad de Potencia [dB]
plt.figure(2, figsize=(10, 4.5))
plt.plot(ff, res_a['S_out_db'], label=r'$s_Q = Q_{B, V_R}(s_R)$ (ADC out)', color='C0', lw=0.9)
plt.plot(ff, S_orig_db, label=r'$s$ (analog)', color='orange', linestyle=':', alpha=0.7)
plt.plot(ff, res_a['S_in_db'], label=r'$s_R = s + n$ (ADC in)', color='green', linestyle=':', alpha=0.7)
graficar_pisos(plt.gca(), res_a['piso_analog'], res_a['piso_digital'])
plt.title(f'Señal muestreada por un ADC de 4 bits - $\pm V_R = {VF:.1f}$ V - q = {res_a["q"]:.3f} V')
plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Densidad de Potencia [dB]')
plt.xlim(0, fs/2)
plt.ylim(-85, 5)
plt.grid(True)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Análisis de la Figura 2 (Espectro):**
En el espectro de potencia se observa que el pico de la senoidal aparece en $1\text{ Hz}$ alcanzando los $0\text{ dB}$ (nuestra referencia de $1\text{ W}$). Por debajo se aprecian los pisos de ruido:
Como en este caso $k_n = 1$, la potencia del ruido analógico es igual a la de cuantización, por lo que ambos pisos teóricos coinciden exactamente en $-55.8\text{ dB}$. Al sumarse estas dos fuentes de igual potencia, el ruido total a la salida se duplica, lo que hace que la relación señal a ruido disminuya aproximadamente $3\text{ dB}$ respecto a la entrada.

In [ ]:
# GRAFICO 3: Histograma del Error de Cuantizacion
plt.figure(3, figsize=(10, 4.5))
bins_hist = np.linspace(-res_a['q']/2, res_a['q']/2, 11)
plt.hist(res_a['ruido_cuantizacion'], bins=bins_hist, color='C0', edgecolor='white', lw=0.5)

# Altura esperada para 1000 muestras en 10 bines: 100 muestras
altura_teorica = N / 10
plt.plot([-res_a['q']/2, -res_a['q']/2, res_a['q']/2, res_a['q']/2], [0, altura_teorica, altura_teorica, 0],
         color='red', linestyle='--', lw=1.5, label='Distribucion teorica')
plt.title(f'Ruido de cuantización para 4 bits - $\pm V_R = {VF:.1f}$ V - q = {res_a["q"]:.3f} V')
plt.xlim(-res_a['q']/2 * 1.1, res_a['q']/2 * 1.1)
plt.ylim(0, altura_teorica * 1.25)
plt.grid(False)
plt.tight_layout()
plt.show()

**Análisis de la Figura 3 (Histograma):**
El histograma muestra que el error de cuantización queda contenido entre $-q/2$ y $+q/2$ ($-0.0625\text{ V}$ y $+0.0625\text{ V}$). Al armar 10 bines con las 1000 muestras, la cantidad de valores que cae en cada barra oscila alrededor de un promedio de 100 muestras, lo que concuerda con lo esperado para una distribución uniforme.

### 3.2 Inciso b: Comparación variando los Bits ($B$) y el Ruido Analógico ($k_n$)

A continuación se repite la simulación para las distintas combinaciones de resolución ($B = 4, 8, 16$ bits) y niveles de ruido ($k_n = 0.1, 1, 10$).

In [ ]:
bits_lista = [4, 8, 16]
kn_lista   = [0.1, 1.0, 10.0]

print("="*85)
print(f"{'B [bits]':<10}{'kn':<8}{'q [V]':<12}{'SNR_antes [dB]':<18}{'SNR_desp [dB]':<18}{'Caida SNR [dB]':<15}")
print("="*85)

resultados = {}
for B in bits_lista:
    for kn in kn_lista:
        res = procesar_adc(B=B, kn=kn)
        resultados[(B, kn)] = res
        print(f"{B:<10}{kn:<8}{res['q']:<12.4e}{res['SNR_antes']:<18.2f}{res['SNR_despues']:<18.2f}{res['delta_SNR']:<15.2f}")
print("="*85)

In [ ]:
# FIGURA 4: Comparacion de espectros para B = 4 bits variando kn
fig4, axs4 = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True)
for idx, kn in enumerate(kn_lista):
    res = resultados[(4, kn)]
    ax = axs4[idx]
    ax.plot(ff, res['S_out_db'], label=r'$s_Q$ (ADC out)', color='C0', lw=0.8)
    ax.plot(ff, S_orig_db, label=r'$s$ (analog)', color='orange', linestyle=':', alpha=0.7)
    ax.plot(ff, res['S_in_db'], label=r'$s_R$ (ADC in)', color='green', linestyle=':', alpha=0.6)
    graficar_pisos(ax, res['piso_analog'], res['piso_digital'])
    ax.set_title(f'B = 4 bits | kn = {kn} (q = {res["q"]:.3f} V)')
    ax.set_ylabel('Potencia [dB]')
    ax.set_ylim(-90, 5)
    ax.grid(True, alpha=0.4)
    ax.legend(loc='lower left', fontsize=8)

axs4[-1].set_xlabel('Frecuencia [Hz]')
axs4[-1].set_xlim(0, fs/2)
fig4.suptitle('Figura 4: Densidad de Potencia para B = 4 bits variando kn', fontsize=12, y=0.99)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 5: Comparacion de espectros para B = 8 bits variando kn
fig5, axs5 = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True)
for idx, kn in enumerate(kn_lista):
    res = resultados[(8, kn)]
    ax = axs5[idx]
    ax.plot(ff, res['S_out_db'], label=r'$s_Q$ (ADC out)', color='C0', lw=0.8)
    ax.plot(ff, S_orig_db, label=r'$s$ (analog)', color='orange', linestyle=':', alpha=0.7)
    ax.plot(ff, res['S_in_db'], label=r'$s_R$ (ADC in)', color='green', linestyle=':', alpha=0.6)
    graficar_pisos(ax, res['piso_analog'], res['piso_digital'])
    ax.set_title(f'B = 8 bits | kn = {kn} (q = {res["q"]:.4e} V)')
    ax.set_ylabel('Potencia [dB]')
    ax.set_ylim(-115, 5)
    ax.grid(True, alpha=0.4)
    ax.legend(loc='lower left', fontsize=8)

axs5[-1].set_xlabel('Frecuencia [Hz]')
axs5[-1].set_xlim(0, fs/2)
fig5.suptitle('Figura 5: Densidad de Potencia para B = 8 bits variando kn', fontsize=12, y=0.99)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 6: Comparacion de espectros para B = 16 bits variando kn
fig6, axs6 = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True)
for idx, kn in enumerate(kn_lista):
    res = resultados[(16, kn)]
    ax = axs6[idx]
    ax.plot(ff, res['S_out_db'], label=r'$s_Q$ (ADC out)', color='C0', lw=0.8)
    ax.plot(ff, S_orig_db, label=r'$s$ (analog)', color='orange', linestyle=':', alpha=0.7)
    ax.plot(ff, res['S_in_db'], label=r'$s_R$ (ADC in)', color='green', linestyle=':', alpha=0.6)
    graficar_pisos(ax, res['piso_analog'], res['piso_digital'])
    ax.set_title(f'B = 16 bits | kn = {kn} (q = {res["q"]:.4e} V)')
    ax.set_ylabel('Potencia [dB]')
    ax.set_ylim(-165, 5)
    ax.grid(True, alpha=0.4)
    ax.legend(loc='lower left', fontsize=8)

axs6[-1].set_xlabel('Frecuencia [Hz]')
axs6[-1].set_xlim(0, fs/2)
fig6.suptitle('Figura 6: Densidad de Potencia para B = 16 bits variando kn', fontsize=12, y=0.99)
plt.tight_layout()
plt.show()

In [ ]:
# FIGURA 7: Comparativa Final de Bits (B = 4, 8, 16 con kn = 1.0)
kn_ref = 1.0
bits_comp = [4, 8, 16]
colores = {4: 'tab:blue', 8: 'tab:orange', 16: 'tab:green'}

fig7, (ax_esp, ax_temp) = plt.subplots(2, 1, figsize=(11, 8.5))
zoom_samples = slice(0, 80)

# Senal continua de referencia en el panel temporal
ax_temp.plot(tt[zoom_samples], s[zoom_samples], label='s (analógica continua)',
             color='black', linestyle='--', lw=1.5, alpha=0.8)

for B_val in bits_comp:
    res = resultados[(B_val, kn_ref)]
    c = colores[B_val]
    
    # Espectros superpuestos
    ax_esp.plot(ff, res['S_out_db'], color=c, alpha=0.75, lw=0.9,
                label=f'B = {B_val:2d} bits (Piso: {res["piso_digital"]:.1f} dB)')
    ax_esp.axhline(res['piso_digital'], color=c, linestyle=':', lw=1.2)
    
    # Escalones temporales
    ax_temp.step(tt[zoom_samples], res['s_cuantizada'][zoom_samples],
                 where='mid', color=c, lw=1.2,
                 label=f'B = {B_val:2d} bits (q = {res["q"]*1000:6.2f} mV)')

ax_esp.set_title(f'Comparación Espectral: Impacto de la Resolución del ADC (kn = {kn_ref})', fontsize=11)
ax_esp.set_xlabel('Frecuencia [Hz]')
ax_esp.set_ylabel('Densidad de Potencia [dB]')
ax_esp.set_xlim(0, fs/2)
ax_esp.set_ylim(-160, 8)
ax_esp.grid(True, alpha=0.4)
ax_esp.legend(loc='lower left', fontsize=9)

ax_temp.set_title('Comparación Temporal: Resolución de los Escalones (Zoom 0 a 0.08 s)', fontsize=11)
ax_temp.set_xlabel('Tiempo [segundos]')
ax_temp.set_ylabel('Amplitud [V]')
ax_temp.grid(True, alpha=0.4)
ax_temp.legend(loc='upper right', fontsize=9)

fig7.suptitle('Figura 7: Impacto Directo de la Cantidad de Bits (B = 4, 8 y 16) con kn = 1.0', fontsize=13, y=0.99)
plt.tight_layout()
plt.show()

## 4. Discusión de Resultados

### 4.1 Influencia del Factor $k_n$
Al cambiar el valor de $k_n$, lo que estamos modificando es la relación entre el ruido que ya viene desde la etapa analógica y el ruido que introduce el conversor al redondear la señal.

Cuando usamos un valor chico ($k_n = 0.1$), la señal de entrada viene bastante limpia y con poco ruido analógico. Sin embargo, como el conversor introduce un ruido de cuantización mayor, el piso digital queda ubicado por encima del analógico. En esta situación, el propio ADC es el que degrada notablemente la señal, provocando una caída importante en la relación señal a ruido (alrededor de $10\text{ dB}$). Es decir, el conversor resulta ser el factor que limita la calidad del sistema.

En el caso intermedio ($k_n = 1$), que fue el analizado en el punto a, los dos ruidos tienen la misma potencia, por lo que sus pisos en el espectro quedan en el mismo nivel. Como se suman dos ruidos iguales, la potencia total de ruido a la salida se duplica, lo que hace caer la SNR en aproximadamente $3\text{ dB}$.

Por último, cuando el ruido analógico es grande ($k_n = 10$), este pasa a predominar por completo sobre el de cuantización, ubicando el piso analógico claramente por encima del digital. Lo interesante en este caso es que la SNR casi no cambia antes y después de digitalizar (la caída es menor a medio decibel). Esto tiene sentido porque la señal ya venía muy ruidosa desde el inicio, por lo que el error de redondeo del conversor prácticamente no llega a empeorar la señal.

### 4.2 Influencia de la Resolución en Bits ($B$)
Al variar la cantidad de bits, se observa directamente el efecto de la resolución tanto en el tiempo como en la frecuencia.

En el dominio temporal, con 4 bits el paso de cuantización es grande y la señal queda con saltos muy marcados en forma de escalera. Al subir a 8 bits estos escalones se vuelven mucho más finos, y al llegar a 16 bits la señal cuantizada sigue a la senoidal de forma casi perfecta, siendo imposible notar los saltos a simple vista.

En el dominio de la frecuencia ocurre algo equivalente. Cada bit que agregamos divide el paso de cuantización a la mitad y reduce la potencia del ruido de cuantización, lo que empuja el piso de ruido digital hacia abajo en el espectro a razón de unos $6\text{ dB}$ por cada bit. Esto se aprecia claramente en la Figura 7, donde el piso digital desciende drásticamente al pasar de 4 a 8 bits y luego a 16 bits.

## 5. Conclusiones

1. A partir de los histogramas se pudo comprobar que el error de cuantización se mantiene dentro del rango de $\pm q/2$ y que su comportamiento se aproxima a una distribución uniforme con potencia $q^2/12$.
2. Se observó que el impacto del conversor en la SNR depende fundamentalmente de la relación entre el ruido analógico y el de cuantización. Si la señal analógica es muy limpia ($k_n$ chico), un conversor de pocos bits introduce una degradación considerable, justificando el uso de más bits. En cambio, si el ruido analógico de entrada ya es elevado ($k_n$ grande), el conversor casi no afecta la calidad final y aumentar la cantidad de bits no aporta una mejora apreciable.
3. Incrementar la cantidad de bits reduce el tamaño de los escalones en el dominio temporal y hace descender de forma notable el piso de ruido digital en el espectro, mejorando la fidelidad de la señal digitalizada.